# Task 2: Player Segment Classification

เราจะทำการฝึกใหม่ด้วย train/test ชุดข้อมูลที่ TA พึ่งเปลี่ยน โดยเริ่มจากสูตร **Logistic Regression** และ **class decision weights**
ที่เลือกไว้ก่อนอัปเดตข้อมูล โดยคง preprocessing เดิม แต่ fit ใหม่เฉพาะ training rows ทุกครั้ง
ไม่ทำ Optuna/เลือกโมเดลซ้ำ และไม่ใช้ test labels หรือ sample predictions เป็นคำตอบ
ทำ 5-fold OOF บน development, วัด audit 15% ด้วยสูตรที่ตรึงไว้ แล้ว refit บน train ทั้งหมด

โมเดลสุดท้าย (ขั้นที่ 7) คือ **blend LR 65% + TabM 25% + XGBoost 10%** แล้วเลือกคลาสด้วย decision weights เดิม ผลทำนายตรงกับคอลัมน์ Task 2 ในไฟล์ที่ได้ Kaggle public **0.82618**

## ภาพรวมและวิธีอ่าน notebook

งานนี้จำแนกผู้เล่นเป็น **Casual (0), Competitive Grinder (1), Social (2) และ Whale (3)** โดยใช้ **Macro F1** ซึ่งเฉลี่ย F1 ของแต่ละคลาสให้น้ำหนักเท่ากัน

ใช้ **Logistic Regression** กับ **decision weights** ที่เลือกจากข้อมูลชุดก่อน ไม่ใช่การค้นหา hyperparameters ใหม่ ขั้นที่ 1–6 ใช้เฉพาะ lr ส่วนขั้นที่ 7 ผสม LR กับ TabM (neural network สำหรับข้อมูลตาราง) และ XGBoost

**Process ในการทำงาน**: โหลด/ตรวจข้อมูล **→** เตรียม preprocessing **→** 5-fold OOF บน development **→** audit 15% **→** refit บน train ทั้งหมด **→** blend กับ TabM และ XGBoost **→** ทำนาย test

## 1. เตรียม environment และกำหนดค่าการทดลอง

ส่วนนี้เป็นการ import ไลบรารีที่ใช้ กำหนดโฟลเดอร์ `models/Task2/` สำหรับเก็บผลระหว่างทางและ `submissions/` สำหรับไฟล์ส่ง (หา root ของ repo จาก `TEAM_GUIDE.md` จึงรันได้จากทุกโฟลเดอร์ใน repo) และแสดงเวอร์ชันแพ็กเกจไว้ตรวจสอบ ส่วน `MPLCONFIGDIR` ใช้กำหนดที่เก็บ config และ cache ของ Matplotlib

กลุ่มเราตั้ง `SEED = 342` ไว้ใช้ตอนสุ่มแบ่งข้อมูล และกำหนดให้ใช้ CPU ไม่เกิน 8 threads ส่วนชื่อคลาสเรียงตาม label 0–3

ใน `CONFIG` ยังมีค่าของโมเดลที่เคยทดลองก่อนหน้าอยู่ แต่รอบนี้ใช้สูตรเดิม จึงตั้ง `lgb_trials` และ `cat_trials` เป็น 0 และไม่ได้เรียก Optuna เพื่อจูนใหม่ ที่ output ยังแสดงจำนวน trials เดิม เพราะคำสั่ง print อยู่ก่อนบรรทัด update

ส่วน `NOISE` และ `SUSPECT` เป็นรายการชื่อคอลัมน์ที่เตรียมไว้ ยังไม่ได้ลบออกจากข้อมูล การเลือกใช้คอลัมน์จะดูจาก `spec` ตอนฝึกอีกที

In [1]:
# นำเข้าไลบรารีสำหรับโหลดข้อมูล เตรียม features และฝึกโมเดล
from pathlib import Path
import os, sys, json, time, hashlib, zipfile, warnings, platform, copy, gc
# หา root ของ repo จาก TEAM_GUIDE.md รันได้จากทุกโฟลเดอร์ใน repo
ROOT = Path.cwd().resolve()
while not (ROOT / "TEAM_GUIDE.md").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("ไม่พบ root ของ repo ให้เปิด notebook จากใน repo")
    ROOT = ROOT.parent
OUT = ROOT / "models" / "Task2"   # ไฟล์ผลระหว่างทาง (Git ignore อยู่แล้ว)
SUB = ROOT / "submissions"        # ไฟล์ผลทำนายสำหรับรวมส่ง
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, log_loss)
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import optuna
from importlib.metadata import version

# กำหนด seed และค่าการทดลอง เพื่อให้ตรวจสอบการรันซ้ำได้
SEED = 342
THREADS = min(8, os.cpu_count() or 2)
CONFIG = dict(folds=5, lgb_trials=12, cat_trials=5,
              lgb_iterations=1400, cat_iterations=1100, early_stopping=65,
              rf_trees=220, et_trees=160, audit_size=0.15,
              decision_trials=500, seed=SEED, threads=THREADS)
# ลำดับชื่อคลาส
CLASSES = np.arange(4)
CLASS_NAMES = ["Casual", "Competitive Grinder", "Social", "Whale"]
NOISE = ["random_metric_1", "random_metric_2", "random_metric_3"]
SUSPECT = ["player_type_tag", "engagement_level", "loyalty_tier", "skill_tier"]
# แสดงเวอร์ชันแพ็กเกจและ CONFIG ก่อนปิด trials สำหรับรอบ refresh
VERSIONS = {p: version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn",
    "lightgbm", "catboost", "xgboost", "optuna", "matplotlib", "torch", "tabm", "nbformat", "nbclient", "ipykernel"]}
print("Python", sys.version, "threads", THREADS)
display(pd.Series(VERSIONS, name="version").to_frame())
print(json.dumps(CONFIG, indent=2))
optuna.logging.set_verbosity(optuna.logging.WARNING)
# รอบนี้ใช้สูตรเดิม จึงไม่เปิดการค้นหาพารามิเตอร์ใหม่
CONFIG.update(lgb_trials=0, cat_trials=0)

Python 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)] threads 8


,version
numpy,2.5.4
pandas,3.1.1
scipy,1.18.1
scikit-learn,1.9.1
lightgbm,4.7.0
catboost,1.2.10
xgboost,3.4.1
optuna,5.0.0
matplotlib,3.11.2
torch,2.15.0


{
  "folds": 5,
  "lgb_trials": 12,
  "cat_trials": 5,
  "lgb_iterations": 1400,
  "cat_iterations": 1100,
  "early_stopping": 65,
  "rf_trees": 220,
  "et_trees": 160,
  "audit_size": 0.15,
  "decision_trials": 500,
  "seed": 342,
  "threads": 8
}


## 2. โหลดข้อมูล ตรวจ schema และแบ่งชุดประเมิน

### 2.1 โหลดไฟล์

ส่วนนี้จะหาไฟล์ใน `data/raw/Task2/` ก่อน ถ้ายังมีไฟล์ไม่ครบ จะหา ZIP ในโฟลเดอร์โปรเจกต์หรือ Downloads แล้วดึง `train.csv`, `test.csv` และ `datacard.txt` ของ Task 2 ออกมา

จากนั้นอ่านข้อมูลด้วย `pd.read_csv()` โดยใช้ค่า default และแสดง datacard ไว้ดูความหมายของแต่ละ feature

### 2.2 ตรวจข้อมูลก่อนฝึก

กลุ่มเราใช้ `assert` ตรวจว่า `segment` มีเฉพาะใน train มี label ครบ 0–3 และไม่มีค่าว่าง รวมถึงตรวจว่ามี features 44 คอลัมน์ และคอลัมน์ของ test เรียงตรงกับ train เมื่อเอา target ออก

นอกจากนี้ยังตรวจ ID ว่างหรือซ้ำ, ID ที่ทับกันระหว่าง train/test และแถว features ที่ซ้ำกัน ถ้าพบปัญหาโค้ดจะหยุด เพื่อให้ตรวจข้อมูลก่อนฝึกต่อ แต่การตรวจส่วนนี้ยังไม่ได้ครอบคลุม leakage ทุกกรณี

### 2.3 แบ่ง development และ audit

กลุ่มเรากันข้อมูล **train 15% ไว้เป็น audit** สำหรับประเมินผล ส่วนอีก **85% เป็น development** โดยใช้ **stratify** เพื่อให้สัดส่วนคลาสใกล้เคียงกับข้อมูลเดิม

ข้อมูลที่โหลดมามี **train 104,000** แถว และ **test 25,889** แถว แบ่ง features เป็นตัวเลข 34 ตัวและหมวดหมู่ 10 ตัว ส่วนข้อมูล train แบ่งไว้ดังนี้:

- **Development**: 88,400 แถว
- **Audit**: 15,600 แถว

ในโค้ดยังแบ่ง development เป็น `fit_idx` 70,720 แถว และ `val_idx` 17,680 แถว ตาม workflow เดิม แต่รอบนี้ใช้ development ทั้งหมดทำ OOF 5 folds โดยไม่ได้ใช้ fit/validation คู่นี้จูนพารามิเตอร์ใหม่ ส่วน `fit_idx` ยังใช้เลือกตัวอย่างมาทดสอบ feature builder

ท้ายส่วนนี้บันทึก split manifest ไว้ดูว่าแต่ละแถวถูกแบ่งไปอยู่ชุดไหน และเก็บ SHA-256 ไว้ตรวจสอบว่าใช้ไฟล์ข้อมูลชุดใดในการรัน

In [2]:
# ระบุตำแหน่งข้อมูล และดึงไฟล์จาก ZIP หากยังมีไฟล์ไม่ครบ
DATA = ROOT / "data" / "raw" / "Task2"
ZIP_CANDIDATES = [ROOT / "player-intelligence-system-cpe-342-ml.zip",
    Path.home() / "Downloads" / "player-intelligence-system-cpe-342-ml.zip"]
if not all((DATA / n).exists() for n in ["train.csv", "test.csv", "datacard.txt"]):
    archive = next((p for p in ZIP_CANDIDATES if p.exists()), None)
    if archive is None:
        raise FileNotFoundError("วาง ZIP ที่ project root หรือวางข้อมูลใน data/raw/Task2")
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in ["train.csv", "test.csv", "datacard.txt"]:
            matches = [n for n in z.namelist() if n.endswith("02_multiclass_classification/" + name)]
            assert len(matches) == 1, matches
            (DATA / name).write_bytes(z.read(matches[0]))
# อ่านคำอธิบาย features และไฟล์ train/test
datacard = (DATA / "datacard.txt").read_text(encoding="utf-8")
print(datacard)
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
# ตรวจ target, schema, ID และแถวข้อมูลที่อาจซ้ำหรือทับกัน
assert "segment" in train and "segment" not in test
assert set(train.segment.unique()) == set(CLASSES)
assert not train.segment.isna().any()
FEATURES = [c for c in train if c not in ["segment", "player_id"]]
assert len(FEATURES) == 44
assert list(test.columns) == [c for c in train.columns if c != "segment"]
assert train.player_id.notna().all() and test.player_id.notna().all()
assert train.player_id.is_unique and test.player_id.is_unique
assert not set(train.player_id).intersection(test.player_id)
assert not train[FEATURES].duplicated().any(), "ต้องเปลี่ยนเป็น grouped stratification เมื่อพบแถวซ้ำ"
assert not pd.util.hash_pandas_object(train[FEATURES], index=False).isin(
    pd.util.hash_pandas_object(test[FEATURES], index=False)).any(), "พบ train/test feature overlap"
# แยกชนิด features และแบ่ง development/audit แบบ stratified
RAW_CAT = train[FEATURES].select_dtypes(include=["object", "str", "string", "category"]).columns.tolist()
RAW_NUM = [c for c in FEATURES if c not in RAW_CAT]
y = train.segment.astype(int).to_numpy()
dev_idx, audit_idx = train_test_split(np.arange(len(train)), test_size=CONFIG["audit_size"],
                                    stratify=y, random_state=SEED)
fit_idx, val_idx = train_test_split(dev_idx, test_size=0.20, stratify=y[dev_idx], random_state=SEED + 1)
assert not set(dev_idx) & set(audit_idx)
# บันทึกการแบ่งข้อมูลและ hash ของไฟล์ต้นฉบับ
manifest = pd.DataFrame({"row_index": np.arange(len(train)), "player_id": train.player_id, "split": "development"})
manifest.loc[audit_idx, "split"] = "audit"
manifest["selection_split"] = "audit"
manifest.loc[fit_idx, "selection_split"] = "fit"
manifest.loc[val_idx, "selection_split"] = "validation"
manifest.to_csv(OUT / "task2_split_manifest.csv", index=False)
DATA_HASH = {n: hashlib.sha256((DATA/n).read_bytes()).hexdigest() for n in ["train.csv", "test.csv", "datacard.txt"]}
print("Shapes", train.shape, test.shape, "features", len(RAW_NUM), "numeric", len(RAW_CAT), "categorical")
print("fit / validation / audit:", len(fit_idx), len(val_idx), len(audit_idx))
display(train.head())

TASK 2: PLAYER SEGMENTATION

TARGET VARIABLE: segment (Multi-class)
4-class classification: 0=Casual[1], 1=Grinder[2], 2=Social[3], 3=Whale[4]

FEATURES (44 features):

1. play_frequency (float)
   Average number of days per week the player logs in

2. avg_session_duration (float)
   Average duration of a single play session in minutes

3. total_playtime_hours (float)
   Cumulative lifetime playtime

4. login_streak (float)
   Maximum number of consecutive daily logins

5. days_since_last_login (float)
   Recency metric - days since last login

6. total_spending_thb (float)
   Lifetime spending on in-game purchases[5]

7. avg_monthly_spending (float)
   Average monthly spending

8. spending_frequency (float)
   Number of purchase transactions per month

9. friend_count (float)
   Number of in-game friends

10. team_play_percentage (float)
    Percentage of matches played in groups

11. chat_activity_score (float)
    Chat frequency and volume score

12. friend_invites_sent (float)
    

Shapes (104000, 46) (25889, 45) features 34 numeric 10 categorical
fit / validation / audit: 70720 17680 15600


,player_id,play_frequency,avg_session_duration,total_playtime_hours,login_streak,days_since_last_login,total_spending_thb,avg_monthly_spending,spending_frequency,friend_count,...,platform,device_type,payment_method,language,account_status,player_type_tag,engagement_level,loyalty_tier,skill_tier,segment
0,P129924,5.166703,83.956912,1784.862304,80.0,94.345964,540.168064,6404.765652,24.400908,6.0,...,NaN,Laptop,Card,NaN,Dormant,Casual,Low,Silver,Silver,2
1,P061398,2.363935,26.686070,4180.785530,91.0,97.429341,26997.309328,NaN,20.242690,32.0,...,Mobile,Tablet,NaN,FR,Dormant,Casual,Mid,Platinum,Bronze,3
2,P023935,34.709108,122.504325,1382.249476,106.0,4.009090,62175.578032,2284.189836,10.729244,33.0,...,PC,Laptop,NaN,DE,Dormant,Competitive,High,Platinum,Diamond,3
3,P091426,11.304823,174.052106,2544.019165,43.0,NaN,28710.877472,NaN,25.437711,122.0,...,Mobile,Laptop,Gift,FR,Dormant,Casual,High,Silver,Silver,1
4,P058826,24.997414,61.008515,3285.259530,82.0,25.814744,33666.539627,1600.569528,10.450822,151.0,...,Console,Tablet,NaN,KR,Active,Competitive,High,Gold,Diamond,1


## 3. Features, preprocessing และตัวช่วยฝึกโมเดล

### 3.1 Feature builder

ฟังก์ชัน `feature_frame()` ใช้เตรียม features ก่อนเข้าโมเดล โดยเลือกคอลัมน์ตาม `FEATURES` ซึ่งไม่รวม `player_id` และ `segment` ส่วน `preferred_game_mode` จะเปลี่ยนจากรหัสตัวเลขให้เป็นข้อมูลหมวดหมู่

ถ้าตั้ง `engineer=True` จะเพิ่ม features เช่น จำนวนค่าว่างในแต่ละแถว อัตราส่วนการเล่นและการใช้จ่าย รวมถึงค่าที่คำนวณร่วมกันจากหลายคอลัมน์ และค่า `log1p` สำหรับตัวเลขบางตัว ส่วนการหารผ่านฟังก์ชัน `ratio()` จะให้เป็น missing ถ้าตัวหารเป็น 0 หรือติดลบ และเปลี่ยนค่า infinity เป็น `NaN`

จากนั้นเลือกหรือตัดคอลัมน์ตาม `keep` และ `drop` เติมค่าว่างของข้อมูลหมวดหมู่ด้วยข้อความ `__MISSING__` และแปลงข้อมูลตัวเลขเป็น `float`

แต่รอบนี้กลุ่มเราใช้สูตรเดิมที่ตั้ง `engineer=False` จึงไม่ได้เพิ่ม features ที่คำนวณไว้ในส่วนนี้

In [3]:
# กำหนด spec เริ่มต้น โดยยังไม่เปิด feature engineering
BASE_SPEC = {"engineer": False, "drop": [], "keep": None}
def feature_frame(raw, spec):
    x = raw[FEATURES].copy()
    x["preferred_game_mode"] = x.preferred_game_mode.map(lambda v: str(int(v)) if pd.notna(v) else np.nan)
    def ratio(a, b):
        return a.div(b.where(b > 0)).replace([np.inf, -np.inf], np.nan)
    # สร้าง ratios และ interaction features เฉพาะเมื่อเปิด engineer=True
    if spec["engineer"]:
        count_cols = [c for c in FEATURES if c not in spec["drop"]]
        x["missing_feature_count"] = x[count_cols].isna().sum(axis=1)
        x["missing_numeric_count"] = x[[c for c in RAW_NUM if c not in spec["drop"]]].isna().sum(axis=1)
        sessions = ratio(x.total_playtime_hours * 60, x.avg_session_duration)
        x["estimated_lifetime_sessions"] = sessions
        x["spending_per_estimated_session"] = ratio(x.total_spending_thb, sessions)
        x["lifetime_spending_per_account_day"] = ratio(x.total_spending_thb, x.account_age_days)
        x["monthly_basket_value"] = ratio(x.avg_monthly_spending, x.spending_frequency)
        x["monthly_to_lifetime_spending"] = ratio(x.avg_monthly_spending, x.total_spending_thb)
        x["playtime_per_account_day"] = ratio(x.total_playtime_hours, x.account_age_days)
        x["friend_invites_per_account_day"] = ratio(x.friend_invites_sent, x.account_age_days)
        x["social_chat_team_proxy"] = x.chat_activity_score * x.team_play_percentage / 100
        x["social_friend_team_proxy"] = x.friend_count * x.team_play_percentage / 100
        x["gift_per_friend"] = ratio(x.gifts_sent_received, x.friend_count)
        x["ranked_playtime_proxy"] = x.total_playtime_hours * x.ranked_participation_rate / 100
        x["ranked_win_activity_proxy"] = x.ranked_participation_rate * x.win_rate_ranked / 100
        x["tournaments_per_account_day"] = ratio(x.tournament_entries, x.account_age_days)
        x["frequency_recency_proxy"] = x.play_frequency / (1 + x.days_since_last_login)
        x["achievement_per_account_day"] = ratio(x.achievement_completion_rate, x.account_age_days)
        x["collection_per_playtime_hour"] = ratio(x.collection_progress, x.total_playtime_hours)
        x["spending_per_playtime_hour"] = ratio(x.total_spending_thb, x.total_playtime_hours)
        x["cosmetic_spending_proxy"] = x.avg_monthly_spending * x.item_type_preference_cosmetic
        x["social_spending_proxy"] = x.avg_monthly_spending * x.item_type_preference_social
        x["performance_spending_proxy"] = x.avg_monthly_spending * x.item_type_preference_performance
        for c in ["total_spending_thb", "avg_monthly_spending", "total_playtime_hours", "friend_count",
                  "spending_frequency", "days_since_last_login", "account_age_days"]:
            x["log1p_" + c] = np.log1p(x[c].where(x[c] >= 0))
    # เลือกคอลัมน์ตาม spec และจัดการชนิดข้อมูลก่อนส่งเข้า preprocessing
    x = x.drop(columns=spec["drop"], errors="ignore")
    if spec.get("keep") is not None: x = x[spec["keep"]]
    cat = x.select_dtypes(include=["object", "str", "string", "category"]).columns.tolist()
    for c in cat: x[c] = x[c].fillna("__MISSING__").astype(str).astype(object)
    for c in x.columns.difference(cat): x[c] = pd.to_numeric(x[c], errors="raise").astype(float)
    return x.replace([np.inf, -np.inf], np.nan), cat



### 3.2 Preprocessing และฟังก์ชันคะแนน

ส่วนนี้เตรียมข้อมูลให้เหมาะกับแต่ละโมเดล โดย `NativePrep` ใช้กับ LightGBM และ CatBoost สำหรับ LightGBM จะใช้ `OrdinalEncoder` แปลงหมวดหมู่เป็นตัวเลข ถ้าเจอหมวดหมู่ที่ไม่มีในข้อมูลฝึกจะให้ค่าเป็น -1 ส่วน CatBoost จะรับข้อมูลหมวดหมู่พร้อมรายชื่อคอลัมน์โดยตรง

สำหรับโมเดลอื่น ฟังก์ชัน `make_preprocessor()` จะเติมตัวเลขที่ว่างด้วย median และเพิ่มคอลัมน์บอกตำแหน่งที่เคยเป็นค่าว่าง ส่วนข้อมูลหมวดหมู่ใช้ `OneHotEncoder` โดยไม่ให้เกิด error เมื่อเจอหมวดหมู่ใหม่ ถ้าเป็น Logistic Regression จะใช้ `StandardScaler` ปรับสเกลตัวเลขหลังเติมค่าว่างด้วย

ค่าที่ใช้เติมข้อมูล การปรับสเกล และการเข้ารหัส จะเรียนรู้จากข้อมูลฝึกของแต่ละรอบเท่านั้น แล้วนำไปใช้กับข้อมูลประเมิน เพื่อไม่ให้ข้อมูลจาก audit หรือ fold ที่กันไว้เข้ามามีผลตอนฝึก

ส่วน `macro()` ใช้เลือกคลาสที่มี probability สูงสุดแล้วคำนวณ Macro F1 และ `lgb_macro()` เตรียมคะแนนนี้ให้อยู่ในรูปแบบที่ LightGBM ใช้ประเมินระหว่างฝึกได้

In [4]:
# เตรียมข้อมูลสำหรับโมเดล native โดย fit encoder เฉพาะข้อมูลฝึก
class NativePrep:
    def __init__(self, family): self.family = family
    def fit_transform(self, x, cat):
        self.cat = list(cat)
        if self.family == "lgb" and self.cat:
            self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            self.encoder.fit(x[cat])
        return self.transform(x)
    def transform(self, x):
        z = x.copy()
        if self.family == "lgb" and self.cat: z[self.cat] = self.encoder.transform(z[self.cat]).astype(np.int32)
        return z

# เติมตัวเลขว่าง เข้ารหัสหมวดหมู่ และเพิ่ม scaling เมื่อใช้ Logistic Regression
def make_preprocessor(x, cat, scale=False):
    num = [c for c in x if c not in cat]
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if scale: num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("numeric", Pipeline(num_steps), num),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32), cat)
    ], sparse_threshold=1.0)

# คำนวณ Macro F1 จาก argmax ของ probability ดิบ
def macro(y_true, p): return float(f1_score(y_true, np.argmax(p, axis=1), labels=CLASSES, average="macro", zero_division=0))
def lgb_macro(y_true, p): return "macro_f1", macro(y_true, p), True



### 3.3 ตัวช่วยฝึกโมเดล

ฟังก์ชัน `fit_model()` ใช้รวมขั้นตอนเตรียมข้อมูลและฝึกโมเดลไว้ด้วยกัน โดยรับชื่อโมเดล แถวที่ใช้ฝึก ชุดสำหรับ early stopping (ถ้ามี) และค่าการตั้งค่าต่าง ๆ แล้วสร้าง preprocessing กับโมเดลใหม่ในแต่ละรอบ

ในฟังก์ชันมีโค้ดรองรับ LightGBM, CatBoost, XGBoost, Logistic Regression, Random Forest และ Extra Trees จาก workflow เดิม แต่รอบนี้กลุ่มเราเรียกใช้เฉพาะ `family='lr'` หรือ Logistic Regression

สำหรับ Logistic Regression จะเติมค่าว่างด้วย median ปรับสเกลตัวเลข และทำ one-hot encoding กับข้อมูลหมวดหมู่ ก่อนฝึกด้วยค่าต่อไปนี้:

- `C=0.5`
- `max_iter=1600`
- `solver='lbfgs'`

ส่วน `params` ของ lr ใน recipe เป็น dict ว่าง จึงใช้ค่าที่กำหนดไว้ข้างต้นโดยไม่มีค่าอื่นมาทับ

รอบนี้ส่ง `stop_rows=None` จึงไม่ได้ใช้ส่วน early stopping ที่เตรียมไว้สำหรับ boosting models ส่วน `rounds=0` ของ lr เป็นค่าที่ฟังก์ชันกำหนดไว้ ไม่ได้หมายความว่าไม่ได้ฝึกโมเดล

เมื่อฝึกเสร็จ ฟังก์ชันจะคืน `bundle` ที่เก็บทั้งโมเดล preprocessing ที่ fit แล้ว และรายละเอียดการตั้งค่าไว้ใช้ตอนทำนายต่อ

In [5]:
# สร้าง preprocessing และโมเดลใหม่จาก training rows ของรอบนั้น
def fit_model(family, fit_rows, stop_rows, spec, params=None, fixed_iterations=None):
    params = copy.deepcopy(params or {})
    x, cat = feature_frame(train.iloc[fit_rows], spec)
    native = family in ["lgb", "cat"]
    prep = NativePrep(family) if native else make_preprocessor(x, cat, scale=family == "lr")
    xt = prep.fit_transform(x, cat) if native else prep.fit_transform(x)
    # ถ้ามีชุด early stopping ให้ใช้ preprocessing ที่ fit จาก training rows แล้ว
    stop_x = None
    if stop_rows is not None:
        sx, _ = feature_frame(train.iloc[stop_rows], spec)
        stop_x = prep.transform(sx)
    # สาขา LightGBM: กำหนดพารามิเตอร์และ early stopping เมื่อมี stop_rows
    if family == "lgb":
        args = dict(objective="multiclass", num_class=4, n_estimators=CONFIG["lgb_iterations"],
            learning_rate=.045, num_leaves=31, max_depth=-1, min_child_samples=45,
            colsample_bytree=.9, subsample=.9, subsample_freq=1, reg_lambda=3.,
            verbosity=-1, n_jobs=THREADS, random_state=SEED, deterministic=True,
            force_col_wise=True, metric="None", importance_type="gain")
        args.update(params)
        if fixed_iterations is not None: args["n_estimators"] = fixed_iterations
        model = LGBMClassifier(**args)
        kw = dict(categorical_feature=cat)
        if stop_x is not None:
            kw.update(eval_set=[(stop_x, y[stop_rows])], eval_metric=lgb_macro,
                      callbacks=[early_stopping(CONFIG["early_stopping"], verbose=False), log_evaluation(0)])
        model.fit(xt, y[fit_rows], **kw)
        rounds = model.best_iteration_ or args["n_estimators"]
    # สาขา CatBoost: ส่ง categorical features ให้โมเดลโดยตรง
    elif family == "cat":
        args = dict(loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
            iterations=CONFIG["cat_iterations"], depth=6, learning_rate=.07, l2_leaf_reg=5.,
            random_seed=SEED, thread_count=THREADS, verbose=False, allow_writing_files=False)
        args.update(params)
        if fixed_iterations is not None: args["iterations"] = fixed_iterations
        if args.get("auto_class_weights") not in [None, "None"] or args.get("class_weights") is not None:
            args["eval_metric"] += ";use_weights=false"
        model = CatBoostClassifier(**args)
        kw = dict(cat_features=cat)
        if stop_x is not None:
            kw.update(eval_set=(stop_x, y[stop_rows]), early_stopping_rounds=CONFIG["early_stopping"], use_best_model=True)
        model.fit(xt, y[fit_rows], **kw)
        rounds = model.tree_count_
    # สาขา XGBoost: ใช้ข้อมูลที่ผ่าน preprocessing แล้ว
    elif family == "xgb":
        args = dict(objective="multi:softprob", num_class=4, n_estimators=1000, max_depth=5,
            learning_rate=.06, min_child_weight=6, subsample=.85, colsample_bytree=.9,
            reg_lambda=5., tree_method="hist", n_jobs=THREADS, random_state=SEED, eval_metric="mlogloss")
        args.update(params)
        if fixed_iterations is not None: args["n_estimators"] = fixed_iterations
        if stop_x is not None: args["early_stopping_rounds"] = CONFIG["early_stopping"]
        model = XGBClassifier(**args)
        model.fit(xt, y[fit_rows], **({"eval_set": [(stop_x, y[stop_rows])], "verbose": False} if stop_x is not None else {}))
        rounds = model.best_iteration + 1 if stop_x is not None else args["n_estimators"]
    else:
        # รอบ refresh นี้เรียกใช้ lr; rf และ et เป็นตัวช่วยจาก workflow เดิม
        if family == "lr":
            model = LogisticRegression(C=.5, max_iter=1600, solver="lbfgs", **params)
        elif family == "rf":
            model = RandomForestClassifier(n_estimators=CONFIG["rf_trees"], min_samples_leaf=3, max_depth=20,
                        max_features="sqrt", n_jobs=THREADS, random_state=SEED, **params)
        elif family == "et":
            model = ExtraTreesClassifier(n_estimators=CONFIG["et_trees"], min_samples_leaf=2, max_depth=20,
                        max_features="sqrt", n_jobs=THREADS, random_state=SEED, **params)
        else: raise ValueError(family)
        model.fit(xt, y[fit_rows]); rounds = 0
    # เก็บโมเดลและ preprocessing ที่ fit แล้วไว้ใน bundle เดียวกัน
    return {"family": family, "spec": copy.deepcopy(spec), "params": params, "prep": prep,
            "model": model, "rounds": int(rounds), "columns": x.columns.tolist()}



### 3.4 ตรวจ probability และทดสอบ feature builder

ฟังก์ชัน `predict_model()` เตรียม features ตาม `spec` แล้วใช้ preprocessing ที่ fit ไว้กับโมเดลเพื่อทำนาย probability ของทั้ง 4 คลาส

จากนั้นตรวจว่าลำดับคลาสถูกต้อง ผลทำนายมีขนาด `(จำนวนแถว, 4)` ไม่มีค่า `NaN` หรือ infinity และ probability ของแต่ละแถวรวมกันได้ประมาณ 1

ส่วนท้ายใช้ข้อมูลตัวอย่างทดสอบว่า feature builder ไม่ได้นำ `segment` หรือ `player_id` มาเป็น input และยังสร้าง features ได้เหมือนเดิมเมื่อเอา target ออก นอกจากนี้ยังตรวจว่าถ้า `account_age_days` เป็น 0 ค่า ratio ที่ใช้คอลัมน์นี้เป็นตัวหารจะได้ missing แทนการหารด้วยศูนย์

ตรงนี้เปิด `engineer=True` เพื่อทดสอบฟังก์ชันเท่านั้น ส่วนโมเดลที่ใช้จริงในรอบนี้ยังตั้งเป็น `engineer=False`

เซลล์ส่วนนี้แบ่งมาจากเซลล์เดิมเพื่อให้อ่านง่ายขึ้น โดยไม่ได้เปลี่ยนคำสั่งหรือรันใหม่ ผลตัวอย่างที่แสดงจึงยังเป็นผลจากการรันเดิม

In [6]:
# ทำนายด้วย preprocessing เดิม พร้อมตรวจลำดับคลาสและความถูกต้องของ probabilities
def predict_model(bundle, raw):
    x, _ = feature_frame(raw, bundle["spec"])
    p = bundle["model"].predict_proba(bundle["prep"].transform(x))
    assert np.array_equal(bundle["model"].classes_, CLASSES)
    assert p.shape == (len(raw), 4) and np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1), 1, atol=1e-5)
    return p

# Unit and leakage checks for the deterministic feature builder.
sample_x, _ = feature_frame(train.iloc[fit_idx[:12]], {**BASE_SPEC, "engineer": True})
assert "segment" not in sample_x and "player_id" not in sample_x
same_x, _ = feature_frame(train.iloc[fit_idx[:12]].drop(columns="segment"), {**BASE_SPEC, "engineer": True})
pd.testing.assert_frame_equal(sample_x, same_x)
probe = train.iloc[fit_idx[:1]].copy(); probe["account_age_days"] = 0
probe_x, _ = feature_frame(probe, {**BASE_SPEC, "engineer": True})
assert probe_x.lifetime_spending_per_account_day.isna().all()
display(sample_x.head())

,play_frequency,avg_session_duration,total_playtime_hours,login_streak,days_since_last_login,total_spending_thb,avg_monthly_spending,spending_frequency,friend_count,team_play_percentage,...,cosmetic_spending_proxy,social_spending_proxy,performance_spending_proxy,log1p_total_spending_thb,log1p_avg_monthly_spending,log1p_total_playtime_hours,log1p_friend_count,log1p_spending_frequency,log1p_days_since_last_login,log1p_account_age_days
84964,2.547738,184.797441,2114.768416,NaN,55.982266,NaN,1827.931979,13.673338,NaN,NaN,...,NaN,987.806503,1135.852301,NaN,7.511487,7.657173,NaN,2.686032,4.042740,7.594884
47163,NaN,80.540211,1272.469705,31.0,43.896877,21046.843593,1588.960122,7.066778,67.0,36.428039,...,976.450317,1028.393634,922.860697,9.954553,7.371464,7.149501,4.219508,2.087754,3.804368,6.645091
15708,11.467020,159.281783,739.713371,35.0,66.202389,5943.490852,1804.493349,24.825728,79.0,33.543806,...,1674.177990,1058.390298,590.595327,8.690220,7.498589,6.607614,4.382027,3.251371,4.207709,6.635947
101333,11.929605,165.987871,650.390883,18.0,54.158746,44166.273090,3392.927829,18.874559,56.0,73.409420,...,1838.151604,2966.138930,2290.896203,10.695739,8.129743,6.479110,4.043051,2.989440,4.010215,7.329750
46529,7.213441,132.565479,2109.990371,52.0,30.057450,10396.482553,5708.480968,3.212930,152.0,38.128286,...,992.713944,2145.429765,4988.310306,9.249319,8.649883,7.654912,5.030438,1.438158,3.435839,6.624065


## 4. โหลดสูตรเดิมและประเมินด้วย 5-fold OOF

### 4.1 สูตรที่ใช้ในรอบนี้

`PRIOR_RECIPE` เก็บโมเดลและ decision weights ที่เลือกไว้จากข้อมูลชุดก่อน แม้จะมีรายละเอียดของ LightGBM และ CatBoost อยู่ด้วย แต่รอบนี้กลุ่มเราเลือกใช้เฉพาะ Logistic Regression ผ่าน `BEST_DEFS`

สูตรนี้ตั้ง `engineer=False`, `drop=[]` และ `keep=None` จึงใช้ features เดิมครบ โดยแปลง `preferred_game_mode` เป็นหมวดหมู่ แต่ไม่ได้เพิ่ม engineered features หรือตัดคอลัมน์ใน `NOISE` และ `SUSPECT` ออก

### 4.2 Decision weights

หลังโมเดลทำนาย probability แล้ว ฟังก์ชัน `decide()` จะคูณค่าของแต่ละคลาสด้วย weights เดิม แล้วเลือกคลาสที่ได้ค่าสูงสุด โดย weights ของคลาส 0–3 คือประมาณ:

[1.0000, 1.1146, 1.1753, 1.1241]

ส่วนนี้เป็นการปรับวิธีเลือกคำตอบหลังทำนาย ไม่ใช่ `class_weight` ที่ใช้ตอนฝึก และค่าหลังคูณไม่ได้ถูกปรับให้รวมกันเป็น 1 อีกครั้ง

ส่วน `alpha` และ check scores ที่อยู่ใน recipe เป็นข้อมูลจากรอบก่อน โค้ดรอบนี้ใช้ weights ที่บันทึกไว้โดยตรง ไม่ได้ใช้ alpha คำนวณหรือค้นหา weights ใหม่

### 4.3 OOF predictions

กลุ่มเราใช้ `StratifiedKFold` แบ่ง development เป็น 5 folds โดยแต่ละรอบฝึกจาก 4 folds แล้วทำนายอีก fold ที่กันไว้ ทำจนครบทุกแถว ทั้ง preprocessing และโมเดลจะถูก fit ใหม่จากข้อมูลฝึกของรอบนั้น

คะแนนที่แสดงระหว่างแต่ละ fold ใช้ `macro()` ซึ่งเลือกคลาสจาก probability ดิบ ส่วนคะแนนท้ายเซลล์ใช้ `decide()` จึงเป็นคะแนนหลังใช้ decision weights และไม่ได้คำนวณจากการเฉลี่ยคะแนนดิบทั้ง 5 folds โดยตรง

ท้ายส่วนนี้บันทึกหมายเลข fold, label จริง, prediction และ probability ดิบไว้ตรวจสอบย้อนหลัง ผล OOF หลังใช้ decision weights ได้ Macro F1 เท่ากับ 0.751680 โดยไม่ได้จูน weights เพิ่มจาก OOF ชุดใหม่นี้

In [7]:
# โหลดสูตรและ decision weights ที่เลือกไว้ก่อนเปลี่ยนชุดข้อมูล
PRIOR_RECIPE = json.loads('{"config": {"folds": 5, "lgb_trials": 12, "cat_trials": 5, "lgb_iterations": 1400, "cat_iterations": 1100, "early_stopping": 65, "rf_trees": 220, "et_trees": 160, "audit_size": 0.15, "decision_trials": 500, "seed": 342, "threads": 8}, "versions": {"numpy": "2.5.4", "pandas": "3.1.1", "scipy": "1.18.1", "scikit-learn": "1.9.1", "lightgbm": "4.7.0", "catboost": "1.2.10", "xgboost": "3.4.1", "optuna": "5.0.0", "matplotlib": "3.11.2", "nbformat": "5.11.1", "nbclient": "0.11.0", "ipykernel": "7.4.0"}, "data_sha256": {"train.csv": "a5ea5b1a51aefae5c2e9a3058f4656e60275e3b87b68260c59e6c6d76daf5338", "test.csv": "fe9e8e8617e81be7f3386d0c530de8ac258020e1f8f1eaddf44a868acceda484", "datacard.txt": "d59d141202fc332e9c3f194d5b07a1e96c4d5ea40b75abae29090327e7fd86c7"}, "selected_models": {"lgb": {"family": "lgb", "spec": {"engineer": false, "drop": [], "keep": null}, "params": {"learning_rate": 0.0268699322515408, "num_leaves": 36, "max_depth": 4, "min_child_samples": 153, "reg_lambda": 1.8922142120154977, "reg_alpha": 0.19370984072731495, "colsample_bytree": 0.7288566964962611, "subsample": 0.8177459107153926, "class_weight": {"0": 1.0, "1": 1.2539267813728665, "2": 1.395659571782427, "3": 1.5986073171321786}}, "selection_name": "optuna_lgb_11"}, "cat": {"family": "cat", "spec": {"engineer": false, "drop": [], "keep": null}, "params": {"auto_class_weights": "SqrtBalanced"}, "selection_name": "cat_sqrt_balanced"}, "lr": {"family": "lr", "spec": {"engineer": false, "drop": [], "keep": null}, "params": {}, "selection_name": "baseline_lr"}}, "iterations": {"lgb": 623, "cat": 486, "lr": 0}, "decision": {"source": "lr", "alpha": 0.9, "weights": [1.0, 1.1145828276911305, 1.175343805006587, 1.1241432712795814], "adjusted": true, "raw_check_f1": 0.7496734146075823, "adjusted_check_f1": 0.7515691646764835}, "domain_auc": 0.5015345371908275}')
DECISION = PRIOR_RECIPE['decision']
BEST_DEFS = {'lr': PRIOR_RECIPE['selected_models']['lr']}
FINAL_ROUNDS = {'lr': 0}
DECISION_WEIGHTS = np.asarray(DECISION['weights'], dtype=float)
# ปรับกฎเลือกคลาสด้วย weights เดิม โดยไม่ปรับ probability กลับให้รวมเป็น 1
def decide(p):
    return np.argmax(p * DECISION_WEIGHTS, axis=1)
# เตรียมพื้นที่เก็บ OOF probabilities และหมายเลข fold ของ development
yd = y[dev_idx]
oof_p = np.zeros((len(dev_idx), 4))
fold_ids = np.full(len(dev_idx), -1)
cv_rows = []
definition = BEST_DEFS['lr']
# ฝึกใหม่ในแต่ละ fold และทำนายเฉพาะแถวที่กันไว้
for fold, (a, b) in enumerate(StratifiedKFold(CONFIG['folds'], shuffle=True, random_state=SEED).split(dev_idx, yd)):
    start = time.perf_counter()
    bundle = fit_model('lr', dev_idx[a], None, definition['spec'], definition['params'])
    oof_p[b] = predict_model(bundle, train.iloc[dev_idx[b]])
    fold_ids[b] = fold
    cv_rows.append({'family':'lr','fold':fold,'macro_f1':macro(yd[b],oof_p[b]),'rounds':0,
                    'seconds':time.perf_counter()-start})
    print('OOF fold', fold+1, 'Macro F1', cv_rows[-1]['macro_f1'], flush=True)
# ตรวจว่าได้ OOF ครบทุกแถว แล้วคำนวณคะแนนหลังใช้ decision weights
assert (fold_ids >= 0).all()
oof_pred = decide(oof_p)
oof_score = float(f1_score(yd, oof_pred, labels=CLASSES, average='macro', zero_division=0))
# บันทึก OOF predictions, probabilities และคะแนนดิบของแต่ละ fold
oof_export = pd.DataFrame({'row_index':dev_idx,'player_id':train.iloc[dev_idx].player_id.to_numpy(),
                           'segment':yd,'fold':fold_ids,'prediction':oof_pred})
for cls in CLASSES:
    oof_export[f'lr_p{cls}'] = oof_p[:,cls]
oof_export.to_csv(OUT/'task2_development_oof.csv',index=False)
pd.DataFrame(cv_rows).to_csv(OUT/'task2_cv_results.csv',index=False)
pd.DataFrame([{'name':'frozen_prior_lr','family':'lr','macro_f1':oof_score}]).to_csv(OUT/'task2_candidate_results.csv',index=False)
print('Fixed recipe OOF Macro F1:',oof_score)

OOF fold 1 Macro F1 0.746584082029607


OOF fold 2 Macro F1 0.7484124409982005


OOF fold 3 Macro F1 0.7498070575526097


OOF fold 4 Macro F1 0.7498688176741771


OOF fold 5 Macro F1 0.7597677272921927


Fixed recipe OOF Macro F1: 0.7516800661196537


## 5. ประเมิน audit set และช่วง bootstrap

หลังจากกำหนดสูตรที่จะใช้แล้ว กลุ่มเราฝึก Logistic Regression บน development ทั้งหมด แล้วนำไปทำนาย audit 15% ที่กันไว้ โดยใช้ preprocessing ที่ fit จาก development ไม่ได้ fit เพิ่มจากข้อมูล audit

จากนั้นแสดง precision, recall และ F1 ของแต่ละคลาส เพื่อดูว่าโมเดลทำได้ดีหรือพลาดในกลุ่มไหน พร้อมบันทึกคำทำนายรายแถวไว้ตรวจสอบ

### ประมาณช่วงคะแนนด้วย stratified bootstrap

ส่วนนี้ใช้ดูว่าคะแนนเปลี่ยนไปแค่ไหนเมื่อสุ่มตัวอย่าง audit ใหม่ โดยสุ่มแบบใส่คืนแยกภายในแต่ละคลาส ให้จำนวนตัวอย่างแต่ละคลาสเท่าเดิม ทำทั้งหมด 400 รอบ แล้วคำนวณ Macro F1 ในแต่ละรอบ

การคำนวณใช้คำทำนาย audit ที่มีอยู่แล้ว ไม่ได้ฝึกโมเดลใหม่ 400 ครั้ง จากนั้นนำเปอร์เซ็นไทล์ที่ 2.5 และ 97.5 มาสร้างช่วง bootstrap 95%

ช่วงนี้สะท้อนความแปรปรวนจากการสุ่มตัวอย่าง audit เท่านั้น ไม่ได้รวมผลจากการเปลี่ยนโมเดลหรือแบ่งข้อมูลฝึกใหม่ทั้งหมด

ผลที่ได้คือ Audit Macro F1 เท่ากับ 0.758128 และช่วง bootstrap ประมาณ [0.751099, 0.764778] โดยเป็นคะแนนประเมินภายใน ไม่ใช่คะแนน Kaggle และรอบนี้ไม่ได้ใช้ผล audit กลับไปเลือกสูตรใหม่

In [8]:
# ฝึกบน development ทั้งหมด แล้วประเมิน audit ด้วย decision weights เดิม
audit_model = fit_model('lr', dev_idx, None, definition['spec'], definition['params'])
audit_probability = predict_model(audit_model, train.iloc[audit_idx])
audit_prediction = decide(audit_probability)
audit_score = float(f1_score(y[audit_idx], audit_prediction, labels=CLASSES, average='macro', zero_division=0))
# บันทึกผลรายคลาสและคำทำนาย audit รายแถว
audit_report = pd.DataFrame(classification_report(y[audit_idx], audit_prediction, labels=CLASSES,
                 target_names=CLASS_NAMES, output_dict=True, zero_division=0)).T
audit_report.to_csv(OUT/'task2_audit_classification_report.csv')
audit_export = pd.DataFrame({'row_index':audit_idx,'player_id':train.iloc[audit_idx].player_id.to_numpy(),
                             'segment':y[audit_idx],'prediction':audit_prediction})
for cls in CLASSES:
    audit_export[f'probability_{cls}'] = audit_probability[:,cls]
audit_export.to_csv(OUT/'task2_audit_predictions.csv',index=False)
# Bootstrap แยกตามคลาสจากคำทำนายเดิม เพื่อประมาณช่วงคะแนนโดยไม่เทรนใหม่
rng = np.random.default_rng(SEED+300)
class_rows = [np.flatnonzero(y[audit_idx] == c) for c in CLASSES]
boot = []
for _ in range(400):
    ix = np.concatenate([rng.choice(g, size=len(g), replace=True) for g in class_rows])
    boot.append(f1_score(y[audit_idx][ix], audit_prediction[ix], labels=CLASSES, average='macro', zero_division=0))
# ใช้เปอร์เซ็นไทล์ 2.5 และ 97.5 เป็นช่วง bootstrap 95%
ci = np.quantile(boot,[.025,.975])
print('Audit Macro F1:',audit_score,'95% bootstrap interval:',ci,flush=True)
display(audit_report)

Audit Macro F1: 0.758128388488463 95% bootstrap interval: [0.75109865 0.76477766]


,precision,recall,f1-score,support
Casual,0.799844,0.835124,0.817103,6144.000000
Competitive Grinder,0.737550,0.758515,0.747885,3905.000000
Social,0.735625,0.705770,0.720388,3154.000000
Whale,0.791414,0.707551,0.747137,2397.000000
accuracy,0.770192,0.770192,0.770192,0.770192
macro avg,0.766108,0.751740,0.758128,15600.000000
weighted avg,0.769971,0.770192,0.769472,15600.000000


## 6. ฝึกโมเดลสุดท้ายและส่งออกผลทำนาย

หลังประเมินเสร็จ กลุ่มเราฝึก Logistic Regression ใหม่ด้วย train ทั้ง 104,000 แถว โดยรวมส่วน audit กลับมาด้วย เพื่อใช้ข้อมูลที่มี label ให้ครบ

ดังนั้นคะแนน audit ที่รายงานก่อนหน้านี้เป็นคะแนนของโมเดลที่ฝึกจาก development ไม่ใช่คะแนนประเมินแยกของโมเดลสุดท้ายที่ใช้ข้อมูล train ทั้งหมดแล้ว

จากนั้นใช้โมเดลและ preprocessing ที่ฝึกไว้ทำนาย test แล้วเลือก label ด้วย decision weights เดิม โดยไม่ได้ใช้ test ในการ fit โมเดลหรือ preprocessing

### ไฟล์ใน models/Task2/

| ไฟล์ | รายละเอียด |
|---|---|
| task2_predictions.csv | row_index, player_id และ prediction หลังใช้ decision weights |
| task2_probabilities.csv | ลำดับแถว, player_id และ probability ดิบทั้ง 4 คลาส ก่อนใช้ weights |
| task2_model_lr.joblib | bundle โมเดล, preprocessing, spec และรายละเอียดจาก fit_model() |
| task2_frozen_config.json | CONFIG, เวอร์ชันแพ็กเกจ, hashes, สูตรเดิม และสถานะว่าไม่ได้ retune |
| task2_run_summary.json | คะแนน OOF/audit, ช่วง bootstrap, ผลรายคลาส และข้อมูลอ้างอิงไฟล์ prediction |

ท้ายเซลล์ตรวจว่า `player_id` และ `row_index` ยังตรงกับ test ทุกแถว ไม่มีคำทำนายว่าง และ label เป็นค่า 0–3 โดยผลที่บันทึกไว้มีทั้งหมด 25,889 แถว

ไฟล์นี้เป็นผลทำนายเฉพาะ Task 2 ยังไม่ได้รวมกับอีก 4 tasks หรือเปลี่ยน `player_id` เป็นรหัส `ANS...` จึงเก็บรหัสผู้เล่นและลำดับแถวเดิมไว้สำหรับนำไปรวมผลต่อ

ส่วน `DECISION` เก็บอยู่ในไฟล์ config และ summary แยกจาก model bundle ถ้าโหลดโมเดลกลับมาใช้ จะต้องใช้ preprocessing และ decision weights ชุดเดิมด้วย เพื่อให้วิธีเลือกคำตอบเหมือนกับรอบนี้

In [9]:
# ฝึกโมเดลสุดท้ายด้วย train ทั้งหมด แล้วทำนาย test
import joblib
final = fit_model('lr', np.arange(len(train)), None, definition['spec'], definition['params'])
test_probability = predict_model(final, test)
prediction = pd.DataFrame({'row_index':np.arange(len(test)),'player_id':test.player_id,
                           'prediction':decide(test_probability)})
path = OUT/'task2_predictions.csv'
prediction.to_csv(path,index=False)
# บันทึก probability ดิบแยกจาก labels ที่ผ่าน decision weights
probability_export = prediction[['row_index','player_id']].copy()
for cls in CLASSES:
    probability_export[f'probability_{cls}'] = test_probability[:,cls]
probability_export.to_csv(OUT/'task2_probabilities.csv',index=False)
# บันทึก model bundle และ config ที่มี decision weights/เวอร์ชัน/ข้อมูลอ้างอิง
joblib.dump(final,OUT/'task2_model_lr.joblib')
refresh_config = {'config':CONFIG,'versions':VERSIONS,'data_sha256':DATA_HASH,
                  'selected_models':BEST_DEFS,'iterations':FINAL_ROUNDS,'decision':DECISION,
                  'mode':'frozen_recipe_dataset_refresh','retuned_on_new_data':False,
                  'prior_recipe_data_sha256':PRIOR_RECIPE['data_sha256']}
(OUT/'task2_frozen_config.json').write_text(json.dumps(refresh_config,indent=2),encoding='utf-8')
# สรุปคะแนนประเมินและรายละเอียดไฟล์ผลลัพธ์ โดยยังไม่ระบุคะแนน Kaggle
summary = {'independent_audit_macro_f1':audit_score,'audit_bootstrap_95_percent':ci.tolist(),
           'audit_per_class':audit_report.loc[CLASS_NAMES].T.to_dict(),
           'oof_macro_f1_selection_biased':oof_score,'decision':DECISION,'final_rounds':FINAL_ROUNDS,
           'output':path.relative_to(ROOT).as_posix(),'output_rows':len(prediction),'output_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
           'data_sha256':DATA_HASH,'mode':'frozen_recipe_dataset_refresh','retuned_on_new_data':False,
           'oof_note':'Fixed recipe from the previous dataset; no new OOF decision tuning.',
           'leaderboard_score':None}
(OUT/'task2_run_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
# ตรวจจำนวนลำดับแถว ID ค่าว่าง และช่วง label ก่อนจบการรัน
assert prediction.player_id.equals(test.player_id) and prediction.row_index.tolist()==list(range(len(test)))
assert not prediction.isna().any().any() and set(prediction.prediction)<={0,1,2,3}
print(json.dumps(summary,indent=2))

{
  "independent_audit_macro_f1": 0.758128388488463,
  "audit_bootstrap_95_percent": [
    0.7510986527411598,
    0.7647776644561239
  ],
  "audit_per_class": {
    "Casual": {
      "precision": 0.7998441153546376,
      "recall": 0.8351236979166666,
      "f1-score": 0.8171032725535473,
      "support": 6144.0
    },
    "Competitive Grinder": {
      "precision": 0.7375498007968128,
      "recall": 0.7585147247119078,
      "f1-score": 0.7478853680090898,
      "support": 3905.0
    },
    "Social": {
      "precision": 0.735624586913417,
      "recall": 0.7057704502219404,
      "f1-score": 0.7203883495145631,
      "support": 3154.0
    },
    "Whale": {
      "precision": 0.7914139057396173,
      "recall": 0.7075511055486025,
      "f1-score": 0.747136563876652,
      "support": 2397.0
    }
  },
  "oof_macro_f1_selection_biased": 0.7516800661196537,
  "decision": {
    "source": "lr",
    "alpha": 0.9,
    "weights": [
      1.0,
      1.1145828276911305,
      1.1753438050065

ผลที่แสดงด้านบนเป็นผลจากการรันกับข้อมูลชุดอัปเดต โดยคะแนน OOF และ audit เป็นคะแนนประเมินภายใน ยังไม่ใช่คะแนนจาก Kaggle

สำหรับ Task 2 กลุ่มเราฝึกโมเดลจากข้อมูลที่ให้มา ไม่ได้ใช้โมเดล pretrained ซึ่งกติกาอนุญาตให้ใช้เฉพาะ Task 4

การประเมินใช้ Macro F1 ของทั้ง 4 คลาส โดยคำนวณด้วย `f1_score(..., labels=[0,1,2,3], average='macro', zero_division=0)`

## 7. Blend: LR 65% + TabM 25% + XGBoost 10%

LR เป็นโมเดลเชิงเส้น จึงผสมกับโมเดลอีกสองตระกูลที่ผิดพลาดต่างจุดกัน แล้วเลือกคลาสด้วย decision weights ชุดเดิม `argmax((0.65·p_LR + 0.25·p_TabM + 0.10·p_XGB) × weights)` น้ำหนัก 65/25/10 ตรึงไว้ก่อนดูผล audit ไม่ได้ค้นหาจาก audit หรือ leaderboard

| โมเดล | รายละเอียด |
|---|---|
| **TabM** | MLP แบบ parameter-efficient ensemble 16 สมาชิก (`tabm` 0.0.3), 2 blocks × 256, dropout 0.1, input 122 คอลัมน์ชุดเดียวกับ LR (median + missing indicator + StandardScaler + one-hot), AdamW lr 0.002, weight decay 0.0003, batch 512, cross-entropy ต่อสมาชิก, seed 20261600; ฝึกจากศูนย์ ไม่ใช้ pretrained |
| **XGBoost** | features ดิบ (ไม่ engineer) + one-hot หมวดหมู่, `max_depth=3`, `min_child_weight=20`, lr 0.04, 780 rounds, `reg_lambda=10`, subsample 0.85, colsample 0.9 |

**การฝึก TabM:** ฝึกบน Kaggle GPU (T4) เลือกจำนวน epoch ด้วย early stopping บน inner split ของ development (ได้ 16 epochs) แล้วฝึก 16 epochs บน development (สำหรับ audit) และบน train ทั้งหมด (สำหรับ test) โค้ดที่ใช้ฝึกอยู่ใน `Task2/tabm_training/` notebook นี้โหลด checkpoint สองไฟล์ใน `Task2/checkpoints/` (ตรวจ SHA256 ก่อน) แล้วทำนายบน CPU ผล CPU ต่างจาก GPU ไม่เกิน 3×10⁻⁷ ไม่ทำให้ label เปลี่ยน

**การเลือก XGBoost:** ทดลอง 4 configs (`max_depth` 3/5 × `min_child_weight` 20/80) บน inner split ของ development แล้วเลือกตัวที่ loss ต่ำสุดพร้อมจำนวนรอบจาก early stopping ไม่ใช้ audit หรือ test ในการเลือก notebook นี้ใช้ config ที่เลือกได้และฝึกใหม่บน CPU

In [10]:
# ตั้งค่า blend และโหลดไลบรารีของ TabM
import torch
import tabm
BLEND_WEIGHTS = {'lr': 0.65, 'tabm': 0.25, 'xgb': 0.10}   # ตรึงไว้ก่อนดูผล audit
TABM_DIR = ROOT / 'Task2' / 'checkpoints'
TABM_MODEL = dict(d_out=4, n_blocks=2, d_block=256, dropout=0.1, k=16, arch_type='tabm',
                  start_scaling_init='random-signs', num_embeddings=None)
# SHA256 ของ checkpoint ที่ได้จากการฝึกบน Kaggle GPU (ตรวจก่อนโหลด)
TABM_SHA = {'development': 'c87791db28137a98d4abb58643b74d079b83fdf28aa8a3785f395c389569b860',
            'final': '6afc4085aed4d6385608d437be3af2fe3b6fa1b9df86f1c78979ab86cfa29ddc'}
XGB_PARAMS = dict(objective='multi:softprob', num_class=4, tree_method='hist', device='cpu', max_bin=128,
                  n_estimators=780, learning_rate=0.04, max_depth=3, min_child_weight=20,
                  reg_lambda=10.0, reg_alpha=0.0, subsample=0.85, colsample_bytree=0.9,
                  random_state=342, n_jobs=2, verbosity=0)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
print('torch', torch.__version__, '| tabm', tabm.__version__)

torch 2.15.0+cpu | tabm 0.0.3


In [11]:
# TabM: ใช้ input เดียวกับ Logistic Regression (median + missing indicator + StandardScaler + one-hot = 122 คอลัมน์)
def tabm_probability(role, bundle, raw):
    path = TABM_DIR / f'tabm_{role}.pt'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == TABM_SHA[role], f'checkpoint {path.name} ไม่ตรงกับที่ฝึกไว้'
    x, _ = feature_frame(raw, bundle['spec'])
    xt = bundle['prep'].transform(x)
    xt = np.asarray(xt.toarray() if hasattr(xt, 'toarray') else xt, dtype=np.float32)
    model = tabm.TabM.make(n_num_features=xt.shape[1], **TABM_MODEL)
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True)['state'])
    model.eval()
    with torch.inference_mode():
        # TabM มี 16 สมาชิก: softmax ของแต่ละสมาชิกแล้วเฉลี่ย
        p = np.concatenate([model(torch.from_numpy(xt[i:i + 512])).softmax(-1).mean(1).numpy()
                            for i in range(0, len(xt), 512)]).astype(np.float64)
    assert p.shape == (len(raw), 4) and np.allclose(p.sum(1), 1, atol=1e-5)
    return p

# XGBoost: features ดิบ (ไม่ engineer), ตัวเลขคง NaN ไว้ให้ XGBoost จัดการ, หมวดหมู่ใช้ one-hot
def xgb_fit(rows):
    x, cat = feature_frame(train.iloc[rows], BASE_SPEC)
    num = [c for c in x if c not in cat]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32).fit(x[cat])
    encode = lambda f: np.column_stack([f[num].to_numpy(dtype=np.float32), encoder.transform(f[cat])])
    model = XGBClassifier(**XGB_PARAMS).fit(encode(x), y[rows])
    assert model.classes_.tolist() == [0, 1, 2, 3]
    return lambda raw: model.predict_proba(encode(feature_frame(raw, BASE_SPEC)[0])).astype(np.float64)

def blend(p_lr, p_tabm, p_xgb):
    p = BLEND_WEIGHTS['lr'] * p_lr + BLEND_WEIGHTS['tabm'] * p_tabm + BLEND_WEIGHTS['xgb'] * p_xgb
    return decide(p)   # ใช้ decision weights เดิมของ LR

### 7.1 ประเมินบน audit

ทุกโมเดลในเซลล์นี้ฝึกจาก development (88,400 แถว) เท่านั้น แล้ววัด Macro F1 บน audit 15,600 แถวด้วย decision weights เดิม

In [12]:
# ประเมินบน audit: ทุกโมเดลฝึกจาก development เท่านั้น
t0 = time.perf_counter()
audit_tabm = tabm_probability('development', audit_model, train.iloc[audit_idx])
audit_xgb = xgb_fit(dev_idx)(train.iloc[audit_idx])
ya = y[audit_idx]
audit_rows = []
for name, pred in [('LR (สูตรเดิม)', decide(audit_probability)),
                   ('TabM', decide(audit_tabm)),
                   ('XGBoost', decide(audit_xgb)),
                   ('LR 75% + TabM 25%', decide(.75 * audit_probability + .25 * audit_tabm)),
                   ('LR 65% + TabM 25% + XGB 10%', blend(audit_probability, audit_tabm, audit_xgb))]:
    audit_rows.append({'model': name, 'audit_macro_f1': f1_score(ya, pred, labels=CLASSES, average='macro', zero_division=0)})
blend_audit = pd.DataFrame(audit_rows)
blend_audit.to_csv(OUT / 'task2_blend_audit.csv', index=False)
display(blend_audit)
audit_blend_prediction = blend(audit_probability, audit_tabm, audit_xgb)
display(pd.DataFrame(classification_report(ya, audit_blend_prediction, labels=CLASSES, target_names=CLASS_NAMES,
                                           output_dict=True, zero_division=0)).T)
print('seconds:', round(time.perf_counter() - t0, 1))

,model,audit_macro_f1
0,LR (สูตรเดิม),0.758128
1,TabM,0.759370
2,XGBoost,0.751823
3,LR 75% + TabM 25%,0.760453
4,LR 65% + TabM 25% + XGB 10%,0.761132


,precision,recall,f1-score,support
Casual,0.803971,0.837077,0.820190,6144.000000
Competitive Grinder,0.743287,0.758515,0.750824,3905.000000
Social,0.734948,0.712112,0.723349,3154.000000
Whale,0.790934,0.713392,0.750165,2397.000000
accuracy,0.773141,0.773141,0.773141,0.773141
macro avg,0.768285,0.755274,0.761132,15600.000000
weighted avg,0.772822,0.773141,0.772487,15600.000000


seconds: 61.7


**ผลลัพธ์บน audit:**

| โมเดล | Audit Macro F1 |
|---|---:|
| LR (สูตรเดิม) | 0.758128 |
| TabM | 0.759370 |
| XGBoost | 0.751823 |
| LR 75% + TabM 25% | 0.760453 |
| **LR 65% + TabM 25% + XGB 10%** | **0.761132** |

- โมเดลเดี่ยวได้คะแนนใกล้กัน แต่การผสมดีกว่าตัวเดี่ยวทุกตัว blend สุดท้ายเพิ่มจาก LR **+0.0030**
- บน 5-fold OOF ของ development (ฝึกครบทุก fold บน GPU) blend ได้ 0.754468 เทียบ LR+TabM 0.753686 และ LR 0.751680
- Social ยังเป็นคลาสที่ยากที่สุด (F1 0.723) ส่วน Whale ได้ precision สูง (0.791) แต่ recall ยังต่ำ (0.713)

**ข้อควรระวัง:** audit ชุดนี้ถูกดูหลายครั้งระหว่างพัฒนา จึงเป็นการเช็คว่าไม่แย่ลง ไม่ใช่การวัดผลอิสระครั้งแรก

### 7.2 ทำนาย test และบันทึกไฟล์

LR และ XGBoost ฝึกบน train ทั้งหมด 104,000 แถว ส่วน TabM ใช้ checkpoint ที่ฝึกบน train ทั้งหมด จากนั้นบันทึก

| ไฟล์ | รายละเอียด |
|---|---|
| `models/Task2/task2_predictions.csv` | row_index, player_id และ prediction ของ blend (เขียนทับไฟล์ LR-only จากขั้นที่ 6) |
| `models/Task2/task2_blend_probabilities.csv` | probability ของ LR, TabM และ XGBoost รายแถว |
| `models/Task2/task2_blend_summary.json` | น้ำหนัก blend, decision weights, คะแนน audit, params, SHA256 ของ checkpoint และเวอร์ชันไลบรารี |
| `submissions/task2.csv` | ไฟล์ส่งตามรูปแบบของทีม `id,task2` (แถวที่ i ของ test ↔ แถวที่ i ของ sample submission) |

In [13]:
# โมเดลสุดท้าย: LR และ XGBoost ฝึกบน train ทั้งหมด, TabM ใช้ checkpoint ที่ฝึกบน train ทั้งหมด
test_tabm = tabm_probability('final', final, test)
test_xgb = xgb_fit(np.arange(len(train)))(test)
blend_prediction = blend(test_probability, test_tabm, test_xgb)
prediction = pd.DataFrame({'row_index': np.arange(len(test)), 'player_id': test.player_id, 'prediction': blend_prediction})
prediction.to_csv(OUT / 'task2_predictions.csv', index=False)
blend_probability = prediction[['row_index', 'player_id']].copy()
for name, p in [('lr', test_probability), ('tabm', test_tabm), ('xgb', test_xgb)]:
    for cls in CLASSES:
        blend_probability[f'{name}_p{cls}'] = p[:, cls]
blend_probability.to_csv(OUT / 'task2_blend_probabilities.csv', index=False)
# ไฟล์ส่งตามรูปแบบของทีม: แถวที่ i ของ test.csv = แถวที่ i ของ sample_submission.csv
sample = pd.read_csv(ROOT / 'data' / 'raw' / 'sample_submission.csv')
assert len(sample) == len(test) == 25889
task2_csv = pd.DataFrame({'id': sample['id'], 'task2': blend_prediction})
SUB.mkdir(exist_ok=True)
task2_csv.to_csv(SUB / 'task2.csv', index=False)
check = pd.read_csv(SUB / 'task2.csv')
assert check.shape == (25889, 2) and set(check.task2) <= {0, 1, 2, 3} and check.id.is_unique
blend_summary = {'blend_weights': BLEND_WEIGHTS, 'decision_weights': DECISION_WEIGHTS.tolist(),
                 'audit_macro_f1': dict(zip(blend_audit.model, blend_audit.audit_macro_f1)),
                 'xgb_params': XGB_PARAMS, 'tabm_model': TABM_MODEL, 'tabm_checkpoint_sha256': TABM_SHA,
                 'versions': {'torch': torch.__version__, 'tabm': tabm.__version__},
                 'changed_vs_lr_only': int((blend_prediction != decide(test_probability)).sum()),
                 'test_class_counts': np.bincount(blend_prediction, minlength=4).tolist()}
(OUT / 'task2_blend_summary.json').write_text(json.dumps(blend_summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved', (SUB / 'task2.csv').relative_to(ROOT).as_posix())
print(json.dumps({k: blend_summary[k] for k in ['changed_vs_lr_only', 'test_class_counts']}))

Saved submissions/task2.csv
{"changed_vs_lr_only": 413, "test_class_counts": [10720, 6576, 5036, 3557]}


**ผลลัพธ์:** ได้คำทำนายครบ 25,889 แถว blend เปลี่ยน label จาก LR-only 413 แถว จำนวนแต่ละคลาสใน test คือ Casual 10,720, Competitive Grinder 6,576, Social 5,036 และ Whale 3,557

คอลัมน์ `task2` นี้ตรงกับคอลัมน์ Task 2 ในไฟล์ที่ส่ง Kaggle และได้คะแนนรวม public **0.82618** ทุกแถว

## สรุปผลการทดลอง

| รายการ | ผล |
|---|---:|
| โมเดลที่ใช้จริง | **LR 65% + TabM 25% + XGBoost 10%** + frozen decision weights |
| Development OOF Macro F1 ของ LR หลังใช้ decision weights | 0.751680 |
| Development OOF Macro F1 ของ blend (fold models จาก GPU) | 0.754468 |
| Audit Macro F1 ของ LR | 0.758128 |
| LR audit bootstrap 95% interval | 0.751099–0.764778 |
| **Audit Macro F1 ของ blend** | **0.761132** |
| จำนวน test rows ที่ส่งออก | 25,889 |

รอบนี้กลุ่มเรานำสูตรเดิมมาฝึกโมเดลใหม่กับข้อมูลชุดอัปเดต โดยไม่ได้ใช้โมเดล pretrained และไม่ได้จูนพารามิเตอร์หรือ decision weights เพิ่มบนข้อมูลชุดนี้ จากนั้นผสมกับ TabM และ XGBoost ที่ฝึกจากศูนย์ ซึ่งเพิ่ม audit Macro F1 จาก 0.758128 เป็น 0.761132

จากผล audit ของ LR คลาส Casual ได้ F1 ประมาณ 0.8171, Competitive Grinder 0.7479, Social 0.7204 และ Whale 0.7471 (blend: 0.8202, 0.7508, 0.7233, 0.7502) โดย Social เป็นกลุ่มที่ได้ F1 ต่ำที่สุด จึงเป็นจุดที่ควรนำไปตรวจคำทำนายที่ผิดเพิ่มเติม

ส่วน Whale มี recall ประมาณ 0.7076 หมายความว่าโมเดลทำนายผู้เล่น Whale ได้ถูกประมาณ 70.76% ของผู้เล่น Whale ทั้งหมดในชุด audit และยังมีบางส่วนที่ถูกทำนายเป็นกลุ่มอื่น

คะแนน OOF และ audit ที่สรุปตรงนี้เป็นคะแนนภายใน ส่วนคอลัมน์ Task 2 จาก blend อยู่ในไฟล์ที่ได้ Kaggle public 0.82618 แต่ Kaggle รายงานเป็นคะแนนรวมของ 5 tasks จึงไม่ทราบคะแนน Kaggle ของ Task 2 อย่างเดียว